# Task 5: Account Anomaly Detection (Unsupervised — $F_3$ Score)

**Objective**: Identify anomalous player accounts (`0` = normal, `1` = anomaly) from 4-period behavioral time-series without training labels, evaluated by **Binary $F_3$ score** (weights recall 9× higher than precision).

### Pipeline Summary
1. **Essential EDA** — Inspect multi-period behavioral shifts, geolocation jumps, and clean distribution gaps separating anomalous subpopulations from normal players.
2. **Stage 1 — Shock Rules & Stationary Clusters**: Flag period-3/4 sudden behavioral/geolocation shifts plus 4 cleanly isolated stationary clusters (`bot`, `lowact`, `lowcomm`, `lowstab`).
3. **Stage 2 — Multi-Period Box Detector**: Flag accounts whose observed values across all 4 periods lie simultaneously inside a tight 3-metric box (`performance_stability`, `kda_std`, `accuracy_std`).
4. **Stage 3 — Missing-Value Bayesian Recovery**: Recover `lowstab` anomalies whose static `behavior_stability` column is `NaN` using a Bayesian log-likelihood ratio over secondary behavioral features (posterior $> 0.10$).

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

def find_data_dir(subdir: str) -> str:
    for p in [os.path.join("..", "cpe342-kaggle-main", subdir), os.path.join("cpe342-kaggle-main", subdir)]:
        if os.path.isdir(p):
            return os.path.abspath(p)
    hits = glob.glob(f"/kaggle/input/**/{subdir}/test.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found for {subdir}. Expected in cpe342-kaggle-main/{subdir}.")

DATA_DIR = find_data_dir("05_anomaly_detection")
OUT_CSV = "task5_predictions.csv"
P = [1, 2, 3, 4]
STATIC = ["behavior_stability", "account_age_days"]

X = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_ids = [f"ANS{i + 1:05d}" for i in range(len(X))]
print(f"Loaded unsupervised test set: {X.shape[0]:,} accounts x {X.shape[1]} columns")

Loaded unsupervised test set: 25,889 accounts x 123 columns


## 1. Essential EDA & Design Decisions

Because no labels are provided, we inspect the data structure directly:
1. **Temporal Structure**: Metrics are recorded across 4 periods (`_1.._4`) plus 2 static account features (`behavior_stability`, `account_age_days`).
2. **Clean Distribution Gaps**: Four features exhibit sharp empty gaps separating small anomalous clusters from the normal population $\rightarrow$ **Decision: use exact gap thresholds instead of black-box outlier scores (which mix anomalies with normal tails).**
3. **Missing `behavior_stability`**: ~5% of accounts have `behavior_stability = NaN`, hiding a small fraction of `lowstab` accounts $\rightarrow$ **Decision: recover them via a Bayesian posterior inside their companion behavioral box.**

In [2]:
mean4 = lambda b: X[[f"{b}_{p}" for p in P]].astype(float).mean(axis=1).values

# Inspect distribution gaps for the 4 stationary clusters
gap_checks = [
    ("login_time_variance (mean)", mean4("login_time_variance"), 1.0, (0.5, 1.5)),
    ("activity_level (mean)", mean4("activity_level"), 0.34, (0.30, 0.40)),
    ("communication_frequency (mean)", mean4("communication_frequency"), 0.17, (0.12, 0.22)),
    ("behavior_stability (static)", X["behavior_stability"].values, 0.55, (0.45, 0.60)),
]
print("Stationary cluster gap check:")
for label, vals, thr, (lo, hi) in gap_checks:
    n_low = int(np.nansum(vals < thr))
    n_gap = int(np.nansum((vals >= lo) & (vals <= hi)))
    print(f"  {label:32s} | < {thr:<4} : {n_low:4d} rows | rows inside gap [{lo}, {hi}]: {n_gap:3d}")

print(f"\nMissing rate of static behavior_stability: {X['behavior_stability'].isna().mean()*100:.2f}%")

Stationary cluster gap check:
  login_time_variance (mean)       | < 1.0  :  207 rows | rows inside gap [0.5, 1.5]:   0
  activity_level (mean)            | < 0.34 :  265 rows | rows inside gap [0.3, 0.4]:   0
  communication_frequency (mean)   | < 0.17 :  263 rows | rows inside gap [0.12, 0.22]:   0
  behavior_stability (static)      | < 0.55 :  246 rows | rows inside gap [0.45, 0.6]:   0

Missing rate of static behavior_stability: 4.86%


## 2. Stage 1 — Period-3/4 Shock Rules & 4 Stationary Clusters

- **Shock / Takeover Rule**: Flags an account if its period-3/4 login coordinates jump $> 5^\circ$ from its period-1/2 home, if $|\text{lat}| > 90$ (invalid coordinate), or if $\ge 2$ behavioral metrics have a period-3/4 change with robust z-score (median/MAD) $> 6$.
- **Stationary Clusters**: Flags `bot` (`login_time_variance < 1.0`), `lowact` (`activity_level < 0.34`), `lowcomm` (`communication_frequency < 0.17`), and `lowstab` (`behavior_stability < 0.55`).

In [3]:
# Geolocation jump: periods 3/4 vs home (mean of periods 1-2)
lat = np.stack([X[f"login_lat_{p}"] for p in P], axis=1).astype(float)
lon = np.stack([X[f"login_lon_{p}"] for p in P], axis=1).astype(float)
bad_lat = (np.abs(lat) > 90).any(axis=1)
lt = np.where(np.abs(lat) > 90, np.nan, lat)
hl, hn = np.nanmean(lt[:, :2], axis=1), np.nanmean(lon[:, :2], axis=1)
jump = np.nan_to_num(
    np.nanmax(np.hypot(lt[:, 2:] - hl[:, None], (lon[:, 2:] - hn[:, None] + 180) % 360 - 180), axis=1)
)

# Behavioral shock: >= 2 metrics with period-3/4 robust z-score > 6
bases = sorted({c[:-2] for c in X.columns if c[-2:] in ("_1", "_2", "_3", "_4")} - {"login_lat", "login_lon"})
Z = []
for b in bases:
    m = X[[f"{b}_{p}" for p in P]].astype(float).values
    ch = np.nanmax(np.abs(m[:, 2:] - np.nanmean(m[:, :2], axis=1, keepdims=True)), axis=1)
    med = np.nanmedian(ch)
    mad = 1.4826 * np.nanmedian(np.abs(ch - med)) + 1e-9
    Z.append((ch - med) / mad)
z2 = np.sort(np.nan_to_num(np.stack(Z, axis=1)), axis=1)[:, -2]

rule = (jump > 5) | bad_lat | (z2 > 6)

# 4 stationary clusters
bot = mean4("login_time_variance") < 1.0
lowact = mean4("activity_level") < 0.34
lowcomm = mean4("communication_frequency") < 0.17
lowstab = X["behavior_stability"].values < 0.55

union_base = rule | bot | lowact | lowcomm | lowstab

for name, mask in [
    ("rule (shock/geo)", rule),
    ("bot", bot),
    ("lowact", lowact),
    ("lowcomm", lowcomm),
    ("lowstab", lowstab),
    ("Stage 1 union", union_base),
]:
    print(f"  {name:18s}: {int(mask.sum()):4d} accounts")

  rule (shock/geo)  :  827 accounts
  bot               :  207 accounts
  lowact            :  265 accounts
  lowcomm           :  263 accounts
  lowstab           :  246 accounts
  Stage 1 union     : 1808 accounts


## 3. Stage 2 — Multi-Period Box Detector (`acckda`)

A 5th stationary anomaly subtype has all observed periods constrained simultaneously to:
- `performance_stability` $\in [0.85, 0.95]$
- `kda_std` $\in [0.70, 1.00]$
- `accuracy_std` $\in [9.0, 12.0]$

Normal accounts almost never satisfy all three multi-period boxes simultaneously.

In [4]:
cols_of = lambda b: [b] if b in STATIC else [f"{b}_{p}" for p in P]


def inbox(b: str, lo: float, hi: float) -> np.ndarray:
    m = X[cols_of(b)].values.astype(float)
    return (((m >= lo) & (m <= hi)) | np.isnan(m)).all(axis=1)


ACCKDA_BOX = {
    "performance_stability": (0.85, 0.95),
    "kda_std": (0.7, 1.0),
    "accuracy_std": (9.0, 12.0),
}
acckda = np.ones(len(X), dtype=bool)
for b, (lo, hi) in ACCKDA_BOX.items():
    acckda &= inbox(b, lo, hi)

acckda_new = acckda & ~union_base
print(f"acckda box total: {int(acckda.sum())} | new outside Stage 1 union: {int(acckda_new.sum())}")

acckda box total: 244 | new outside Stage 1 union: 244


## 4. Stage 3 — Bayesian Recovery of Hidden `lowstab` Accounts

Observed `lowstab` accounts also lie inside `purchase_frequency` $\in [3, 6]$ (all periods) and `account_age_days` $\in [800, 1500]$. For non-union accounts with `behavior_stability = NaN` inside this companion box, we compute a Bayesian posterior using histogram log-likelihood ratios of `mean(login_count)` and `std(purchase_frequency)`, flagging accounts with posterior $> 0.10$ (appropriate for the recall-heavy $F_3$ metric).

In [5]:
bsn = X["behavior_stability"].isna().values
pf_box = inbox("purchase_frequency", 3.0, 6.0)
age_box = inbox("account_age_days", 800.0, 1500.0)

nr = ~union_base & ~bsn
cand = bsn & ~union_base & pf_box & age_box
nor = nr & pf_box & age_box & (X["behavior_stability"].values >= 0.6)

lm = X[[f"login_count_{p}" for p in P]].mean(axis=1).values
ps = X[[f"purchase_frequency_{p}" for p in P]].std(axis=1).values


def logpdf(v: np.ndarray, ref: np.ndarray, bins: np.ndarray) -> np.ndarray:
    h, _ = np.histogram(ref[~np.isnan(ref)], bins=bins)
    h = (h + 0.5) / (h.sum() + 0.5 * len(h)) / np.diff(bins)
    o = np.log(h[np.clip(np.digitize(v, bins) - 1, 0, len(h) - 1)])
    o[np.isnan(v)] = 0.0
    return o


bl = np.linspace(0, 130, 27)
bp = np.linspace(0, 2.5, 26)
LLR = (
    logpdf(lm, lm[lowstab], bl) - logpdf(lm, lm[nor], bl)
    + logpdf(ps, ps[lowstab], bp) - logpdf(ps, ps[nor], bp)
)

# Expected hidden count (~14 based on ~5% missing rate of behavior_stability)
HID = 14.0
prior = HID / cand.sum()
post_lowstab_nan = np.where(cand, 1.0 / (1.0 + np.exp(-(LLR + np.log(prior / (1.0 - prior))))), 0.0)
lsnan_flag = post_lowstab_nan > 0.10

print(f"NaN-behavior_stability candidates in companion box: {int(cand.sum())} | flagged (posterior > 0.10): {int(lsnan_flag.sum())}")

NaN-behavior_stability candidates in companion box: 129 | flagged (posterior > 0.10): 53


## 5. Final Union & Save Predictions

In [6]:
final_pred = (union_base | acckda | lsnan_flag).astype(int)

sub_t5 = pd.DataFrame({"id": test_ids, "pred": final_pred})
sub_t5.to_csv(OUT_CSV, index=False, lineterminator="\n")

print(f"Saved {OUT_CSV} | total rows={len(sub_t5):,} | total flagged anomalies={int(final_pred.sum()):,} ({final_pred.mean()*100:.2f}%)")
sub_t5.head()

Saved task5_predictions.csv | total rows=25,889 | total flagged anomalies=2,104 (8.13%)


,id,pred
0,ANS00001,1
1,ANS00002,0
2,ANS00003,1
3,ANS00004,0
4,ANS00005,0
